# Análise Exploratória — Source

Este notebook explora os dados brutos da **NYC TLC** diretamente da fonte, antes de qualquer ingestão no Data Lake.

O objetivo é responder as seguintes perguntas antes de construir o `etl_bronze.py`:

- Os links dos arquivos estão acessíveis?
- Qual o volume de dados por mês e por tabela?
- A tipagem das colunas é consistente entre os meses?
- Quais as diferenças de schema entre Yellow e Green Taxi?
- Quais colunas obrigatórias estão presentes na fonte?

As conclusões documentadas aqui embasam as decisões de ingestão no `etl_bronze.py`.

**Fonte:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
**Período:** Janeiro a Maio de 2023  
**Tabelas:** Yellow Taxi e Green Taxi

## 1. Importações e configurações

Bibliotecas utilizadas nesta análise:
- `pandas` para leitura e manipulação dos dados
- `requests` para verificar a disponibilidade dos arquivos na fonte
- `logging` para rastreabilidade das operações

In [1]:
import logging
import pandas as pd
import requests
from io import BytesIO

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] exploratory_source - %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S",
)
logger = logging.getLogger("exploratory_source")

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

logger.info("Bibliotecas carregadas com sucesso")

07-06-2026 10:56:59 [INFO] exploratory_source - Bibliotecas carregadas com sucesso


## 2. Configurações da fonte

Definição das URLs base, meses e tabelas que serão analisadas.
O padrão de URL da NYC TLC é previsível e segue o formato:
`{BASE_URL}/{prefixo}_{ano}-{mes}.parquet`

In [2]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
MESES = ["2023-01", "2023-02", "2023-03", "2023-04", "2023-05"]
TABELAS = {
    "yellow_taxi": "yellow_tripdata",
    "green_taxi": "green_tripdata",
}

logger.info(f"Base URL configurada | url={BASE_URL}")
logger.info(f"Meses configurados | meses={MESES}")
logger.info(f"Tabelas configuradas | tabelas={list(TABELAS.keys())}")

07-06-2026 10:57:05 [INFO] exploratory_source - Base URL configurada | url=https://d37ci6vzurychx.cloudfront.net/trip-data
07-06-2026 10:57:05 [INFO] exploratory_source - Meses configurados | meses=['2023-01', '2023-02', '2023-03', '2023-04', '2023-05']
07-06-2026 10:57:05 [INFO] exploratory_source - Tabelas configuradas | tabelas=['yellow_taxi', 'green_taxi']


## 3. Verificação de disponibilidade dos arquivos

Antes de qualquer leitura, verificamos se todos os arquivos estão acessíveis na fonte.
Usamos `requests.head()` para verificar apenas o status HTTP sem baixar o arquivo completo,
o que é mais eficiente para uma verificação de disponibilidade.

In [3]:
logger.info("Iniciando verificacao de disponibilidade dos arquivos")

disponiveis = []
indisponiveis = []

for tabela, prefixo in TABELAS.items():
    for mes in MESES:
        url = f"{BASE_URL}/{prefixo}_{mes}.parquet"
        response = requests.head(url)
        if response.status_code == 200:
            logger.info(f"Arquivo disponivel | tabela={tabela} | mes={mes} | status={response.status_code}")
            disponiveis.append(url)
        else:
            logger.warning(f"Arquivo indisponivel | tabela={tabela} | mes={mes} | status={response.status_code}")
            indisponiveis.append(url)

logger.info(f"Verificacao concluida | disponiveis={len(disponiveis)} | indisponiveis={len(indisponiveis)}")

07-06-2026 10:57:07 [INFO] exploratory_source - Iniciando verificacao de disponibilidade dos arquivos
07-06-2026 10:57:07 [INFO] exploratory_source - Arquivo disponivel | tabela=yellow_taxi | mes=2023-01 | status=200
07-06-2026 10:57:08 [INFO] exploratory_source - Arquivo disponivel | tabela=yellow_taxi | mes=2023-02 | status=200
07-06-2026 10:57:08 [INFO] exploratory_source - Arquivo disponivel | tabela=yellow_taxi | mes=2023-03 | status=200
07-06-2026 10:57:08 [INFO] exploratory_source - Arquivo disponivel | tabela=yellow_taxi | mes=2023-04 | status=200
07-06-2026 10:57:08 [INFO] exploratory_source - Arquivo disponivel | tabela=yellow_taxi | mes=2023-05 | status=200
07-06-2026 10:57:09 [INFO] exploratory_source - Arquivo disponivel | tabela=green_taxi | mes=2023-01 | status=200
07-06-2026 10:57:09 [INFO] exploratory_source - Arquivo disponivel | tabela=green_taxi | mes=2023-02 | status=200
07-06-2026 10:57:09 [INFO] exploratory_source - Arquivo disponivel | tabela=green_taxi | mes=20

## 4. Carregamento dos dados em memória

Todos os arquivos são carregados uma única vez e armazenados em `dfs_por_tabela`.
As células seguintes reutilizam esses dados sem novos downloads,
evitando requisições desnecessárias à fonte.

In [4]:
logger.info("Iniciando carregamento unico de todos os arquivos em memoria")

dfs_por_tabela = {}
tamanhos_por_tabela = {}

for tabela, prefixo in TABELAS.items():
    dfs_por_tabela[tabela] = {}
    tamanhos_por_tabela[tabela] = {}
    for mes in MESES:
        url = f"{BASE_URL}/{prefixo}_{mes}.parquet"
        logger.info(f"Carregando | tabela={tabela} | mes={mes}")
        response = requests.get(url)
        df = pd.read_parquet(BytesIO(response.content))
        dfs_por_tabela[tabela][mes] = df
        tamanhos_por_tabela[tabela][mes] = round(len(response.content) / 1024 / 1024, 2)
        logger.info(f"Carregado | tabela={tabela} | mes={mes} | linhas={df.shape[0]:,} | tamanho={tamanhos_por_tabela[tabela][mes]}MB")

logger.info(f"Carregamento concluido | total_arquivos={len(TABELAS) * len(MESES)}")

07-06-2026 10:57:12 [INFO] exploratory_source - Iniciando carregamento unico de todos os arquivos em memoria
07-06-2026 10:57:12 [INFO] exploratory_source - Carregando | tabela=yellow_taxi | mes=2023-01
07-06-2026 10:57:17 [INFO] exploratory_source - Carregado | tabela=yellow_taxi | mes=2023-01 | linhas=3,066,766 | tamanho=45.46MB
07-06-2026 10:57:17 [INFO] exploratory_source - Carregando | tabela=yellow_taxi | mes=2023-02
07-06-2026 10:57:22 [INFO] exploratory_source - Carregado | tabela=yellow_taxi | mes=2023-02 | linhas=2,913,955 | tamanho=45.54MB
07-06-2026 10:57:22 [INFO] exploratory_source - Carregando | tabela=yellow_taxi | mes=2023-03
07-06-2026 10:57:28 [INFO] exploratory_source - Carregado | tabela=yellow_taxi | mes=2023-03 | linhas=3,403,766 | tamanho=53.53MB
07-06-2026 10:57:28 [INFO] exploratory_source - Carregando | tabela=yellow_taxi | mes=2023-04
07-06-2026 10:57:33 [INFO] exploratory_source - Carregado | tabela=yellow_taxi | mes=2023-04 | linhas=3,288,250 | tamanho=51.

## 5. Volume de dados por tabela e mês

Análise do volume de registros, quantidade de colunas e tamanho real em MB de cada arquivo.

Essa análise nos ajuda a:
- Entender o tamanho do dataset que será ingerido no Bronze
- Identificar variações de volume entre os meses

In [5]:
logger.info("Iniciando analise de volume por tabela e mes")

volumes = []

for tabela in TABELAS.keys():
    for mes in MESES:
        df = dfs_por_tabela[tabela][mes]
        volumes.append({
            "tabela": tabela,
            "mes": mes,
            "linhas": df.shape[0],
            "colunas": df.shape[1],
            "tamanho_mb": tamanhos_por_tabela[tabela][mes],
        })
        logger.info(f"Volume calculado | tabela={tabela} | mes={mes} | linhas={df.shape[0]:,} | tamanho={tamanhos_por_tabela[tabela][mes]}MB")

df_volumes = pd.DataFrame(volumes)

logger.info("Analise de volume concluida")

display(
    df_volumes.set_index(["tabela", "mes"])
    .rename(columns={
        "linhas": "Linhas",
        "colunas": "Colunas",
        "tamanho_mb": "Tamanho (MB)",
    })
    .style
    .set_caption("Volume por Tabela e Mes — Janeiro a Maio 2023")
    .format("{:,}", subset=["Linhas"])
    .format("{:.2f}", subset=["Tamanho (MB)"])
    .set_properties(**{"text-align": "right"})
)

07-06-2026 10:57:55 [INFO] exploratory_source - Iniciando analise de volume por tabela e mes
07-06-2026 10:57:55 [INFO] exploratory_source - Volume calculado | tabela=yellow_taxi | mes=2023-01 | linhas=3,066,766 | tamanho=45.46MB
07-06-2026 10:57:55 [INFO] exploratory_source - Volume calculado | tabela=yellow_taxi | mes=2023-02 | linhas=2,913,955 | tamanho=45.54MB
07-06-2026 10:57:55 [INFO] exploratory_source - Volume calculado | tabela=yellow_taxi | mes=2023-03 | linhas=3,403,766 | tamanho=53.53MB
07-06-2026 10:57:55 [INFO] exploratory_source - Volume calculado | tabela=yellow_taxi | mes=2023-04 | linhas=3,288,250 | tamanho=51.71MB
07-06-2026 10:57:55 [INFO] exploratory_source - Volume calculado | tabela=yellow_taxi | mes=2023-05 | linhas=3,513,649 | tamanho=55.94MB
07-06-2026 10:57:55 [INFO] exploratory_source - Volume calculado | tabela=green_taxi | mes=2023-01 | linhas=68,211 | tamanho=1.36MB
07-06-2026 10:57:55 [INFO] exploratory_source - Volume calculado | tabela=green_taxi | mes

## 6. Verificação de consistência de tipagem por mês

Um ponto crítico antes de construir o ETL é verificar se a tipagem das colunas
é consistente entre os meses. Mudanças de schema entre meses podem causar
falhas silenciosas no pipeline de ingestão.

In [6]:
logger.info("Detalhando inconsistencias | gerando dataframe comparativo")

resultados = []

for tabela in TABELAS.keys():
    schemas = {}
    for mes in MESES:
        schemas[mes] = dict(dfs_por_tabela[tabela][mes].dtypes)

    todas_colunas = set()
    for s in schemas.values():
        todas_colunas.update(s.keys())

    for col in sorted(todas_colunas):
        tipos_por_mes = {mes: str(schemas[mes].get(col, "AUSENTE")) for mes in MESES}
        valores = list(tipos_por_mes.values())
        inconsistente = len(set(valores)) > 1

        if inconsistente:
            row = {"tabela": tabela, "coluna": col}
            row.update(tipos_por_mes)
            resultados.append(row)

df_resultado = pd.DataFrame(resultados)

def highlight_diff(row):
    tipos = row[MESES].values
    return [
        "background-color: #ffeecc" if t == "AUSENTE"
        else "background-color: #ffcccc" if t != tipos[0]
        else ""
        for t in tipos
    ]

logger.info(f"Total de colunas inconsistentes encontradas | total={len(df_resultado)}")

display(
    df_resultado
    .set_index(["tabela", "coluna"])
    .style
    .apply(highlight_diff, subset=MESES, axis=1)
    .set_caption("Inconsistencias de Schema por Tabela e Mes")
    .set_properties(**{"text-align": "left"})
)

07-06-2026 10:58:15 [INFO] exploratory_source - Detalhando inconsistencias | gerando dataframe comparativo
07-06-2026 10:58:15 [INFO] exploratory_source - Total de colunas inconsistentes encontradas | total=9


## 7. Schema detalhado por tabela

Exibimos o schema completo de cada tabela usando Janeiro/2023 como referência.

In [7]:
logger.info("Carregando schema detalhado de cada tabela")

for tabela in TABELAS.keys():
    df = dfs_por_tabela[tabela]["2023-01"]

    schema_df = pd.DataFrame({
        "coluna": df.columns,
        "tipo": df.dtypes.values,
        "nulos": df.isnull().sum().values,
        "pct_nulos": (df.isnull().sum().values / len(df) * 100).round(2),
    })

    logger.info(f"Schema carregado | tabela={tabela} | total_colunas={len(df.columns)}")

    display(
        schema_df
        .set_index("coluna")
        .style
        .set_caption(f"Schema detalhado — {tabela} (2023-01)")
        .format("{:.2f}%", subset=["pct_nulos"])
        .bar(subset=["pct_nulos"], color="#ffcccc", vmin=0, vmax=100)
        .set_properties(**{"text-align": "left"})
    )

07-06-2026 10:58:18 [INFO] exploratory_source - Carregando schema detalhado de cada tabela
07-06-2026 10:58:18 [INFO] exploratory_source - Schema carregado | tabela=yellow_taxi | total_colunas=19


,tipo,nulos,pct_nulos
coluna,,,
VendorID,int64,0,0.00%
tpep_pickup_datetime,datetime64[us],0,0.00%
tpep_dropoff_datetime,datetime64[us],0,0.00%
passenger_count,float64,71743,2.34%
trip_distance,float64,0,0.00%
RatecodeID,float64,71743,2.34%
store_and_fwd_flag,str,71743,2.34%
PULocationID,int64,0,0.00%
DOLocationID,int64,0,0.00%


07-06-2026 10:58:18 [INFO] exploratory_source - Schema carregado | tabela=green_taxi | total_colunas=20


,tipo,nulos,pct_nulos
coluna,,,
VendorID,int64,0,0.00%
lpep_pickup_datetime,datetime64[us],0,0.00%
lpep_dropoff_datetime,datetime64[us],0,0.00%
store_and_fwd_flag,str,4324,6.34%
RatecodeID,float64,4324,6.34%
PULocationID,int64,0,0.00%
DOLocationID,int64,0,0.00%
passenger_count,float64,4324,6.34%
trip_distance,float64,0,0.00%


## 8. Diferenças de schema entre Yellow e Green

Yellow e Green Taxi possuem schemas diferentes. Identificar essas diferenças
é fundamental para planejar a padronização que ocorrerá na camada Silver,
especialmente para as colunas obrigatórias do case.

In [8]:
logger.info("Gerando comparacao de colunas Yellow vs Green")

cols_por_tabela = {}
for tabela in TABELAS.keys():
    todas_colunas = set()
    for mes in MESES:
        todas_colunas.update(dfs_por_tabela[tabela][mes].columns.tolist())
    cols_por_tabela[tabela] = todas_colunas
    logger.info(f"Total de colunas unicas | tabela={tabela} | total={len(todas_colunas)}")

cols_yellow = cols_por_tabela["yellow_taxi"]
cols_green  = cols_por_tabela["green_taxi"]

apenas_yellow = cols_yellow - cols_green
apenas_green  = cols_green - cols_yellow
em_comum      = cols_yellow & cols_green

logger.info(f"Colunas apenas no Yellow | total={len(apenas_yellow)} | colunas={sorted(apenas_yellow)}")
logger.info(f"Colunas apenas no Green  | total={len(apenas_green)} | colunas={sorted(apenas_green)}")
logger.info(f"Colunas em comum         | total={len(em_comum)}")

rows = []
for col in sorted(cols_yellow | cols_green):
    rows.append({
        "coluna": col,
        "yellow_taxi": "✓" if col in cols_yellow else "✗",
        "green_taxi": "✓" if col in cols_green else "✗",
    })

df_colunas = pd.DataFrame(rows)

logger.info("Comparacao concluida")

display(
    df_colunas
    .set_index("coluna")
    .style
    .set_caption("Comparacao de Colunas — Yellow vs Green (todos os meses)")
    .apply(lambda col: [
        "background-color: #ffcccc; color: #842029; font-weight: bold" if v == "✗"
        else "background-color: #d1e7dd; color: #0a3622; font-weight: bold"
        for v in col
    ], subset=["yellow_taxi", "green_taxi"])
    .set_properties(**{"text-align": "center"})
)

07-06-2026 10:58:20 [INFO] exploratory_source - Gerando comparacao de colunas Yellow vs Green
07-06-2026 10:58:20 [INFO] exploratory_source - Total de colunas unicas | tabela=yellow_taxi | total=20
07-06-2026 10:58:20 [INFO] exploratory_source - Total de colunas unicas | tabela=green_taxi | total=20
07-06-2026 10:58:20 [INFO] exploratory_source - Colunas apenas no Yellow | total=4 | colunas=['Airport_fee', 'airport_fee', 'tpep_dropoff_datetime', 'tpep_pickup_datetime']
07-06-2026 10:58:20 [INFO] exploratory_source - Colunas apenas no Green  | total=4 | colunas=['ehail_fee', 'lpep_dropoff_datetime', 'lpep_pickup_datetime', 'trip_type']
07-06-2026 10:58:20 [INFO] exploratory_source - Colunas em comum         | total=16
07-06-2026 10:58:20 [INFO] exploratory_source - Comparacao concluida


,yellow_taxi,green_taxi
coluna,,
Airport_fee,✓,✗
DOLocationID,✓,✓
PULocationID,✓,✓
RatecodeID,✓,✓
VendorID,✓,✓
airport_fee,✓,✗
congestion_surcharge,✓,✓
ehail_fee,✗,✓
extra,✓,✓


## 9. Verificação das colunas obrigatórias do case

O case exige que as seguintes colunas estejam presentes na camada de consumo:
- `VendorID`
- `passenger_count`
- `total_amount`
- `tpep_pickup_datetime` (Yellow) / `lpep_pickup_datetime` (Green)
- `tpep_dropoff_datetime` (Yellow) / `lpep_dropoff_datetime` (Green)

Verificamos se todas estão presentes na fonte.

In [9]:
logger.info("Verificando presenca das colunas obrigatorias do case")

colunas_obrigatorias = {
    "yellow_taxi": [
        "VendorID",
        "passenger_count",
        "total_amount",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
    ],
    "green_taxi": [
        "VendorID",
        "passenger_count",
        "total_amount",
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
    ],
}

rows = []
for tabela, colunas in colunas_obrigatorias.items():
    # usa todos os meses para garantir presença em qualquer mês
    todas_colunas = set()
    for mes in MESES:
        todas_colunas.update(dfs_por_tabela[tabela][mes].columns.tolist())

    for col in colunas:
        presente = col in todas_colunas
        tipo = str(dfs_por_tabela[tabela]["2023-01"][col].dtype) if presente else "N/A"
        rows.append({
            "tabela": tabela,
            "coluna": col,
            "presente": "✓" if presente else "✗",
            "tipo": tipo,
        })
        if presente:
            logger.info(f"Coluna presente | tabela={tabela} | coluna={col} | tipo={tipo}")
        else:
            logger.error(f"Coluna ausente  | tabela={tabela} | coluna={col}")

df_obrigatorias = pd.DataFrame(rows)

logger.info("Verificacao de colunas obrigatorias concluida")

display(
    df_obrigatorias
    .set_index(["tabela", "coluna"])
    .style
    .set_caption("Colunas Obrigatorias do Case — Verificacao na Fonte")
    .apply(lambda col: [
        "background-color: #ffcccc; color: #842029; font-weight: bold" if v == "✗"
        else "background-color: #d1e7dd; color: #0a3622; font-weight: bold"
        for v in col
    ], subset=["presente"])
    .set_properties(**{"text-align": "left"})
)

07-06-2026 10:58:22 [INFO] exploratory_source - Verificando presenca das colunas obrigatorias do case
07-06-2026 10:58:22 [INFO] exploratory_source - Coluna presente | tabela=yellow_taxi | coluna=VendorID | tipo=int64
07-06-2026 10:58:22 [INFO] exploratory_source - Coluna presente | tabela=yellow_taxi | coluna=passenger_count | tipo=float64
07-06-2026 10:58:22 [INFO] exploratory_source - Coluna presente | tabela=yellow_taxi | coluna=total_amount | tipo=float64
07-06-2026 10:58:22 [INFO] exploratory_source - Coluna presente | tabela=yellow_taxi | coluna=tpep_pickup_datetime | tipo=datetime64[us]
07-06-2026 10:58:22 [INFO] exploratory_source - Coluna presente | tabela=yellow_taxi | coluna=tpep_dropoff_datetime | tipo=datetime64[us]
07-06-2026 10:58:22 [INFO] exploratory_source - Coluna presente | tabela=green_taxi | coluna=VendorID | tipo=int64
07-06-2026 10:58:22 [INFO] exploratory_source - Coluna presente | tabela=green_taxi | coluna=passenger_count | tipo=float64
07-06-2026 10:58:22 [

## 10. Conclusões

Com base na análise exploratória da fonte, as seguintes decisões foram tomadas para o `etl_bronze.py` e `etl_silver.py`:

### Ingestão — decisões para o `etl_bronze.py`

| Observação | Decisão |
|---|---|
| Todos os arquivos acessíveis na fonte | Download seguro via `requests` + `boto3` |
| Volumes diferentes por mês | Particionamento por `partition_year` e `partition_month` no S3 |
| Yellow tem volume ~47x maior que Green | Processar tabelas de forma independente no ETL |
| Bronze deve preservar o dado bruto | Sem transformações no `etl_bronze.py`, apenas ingestão |

### Schema — decisões para o `etl_silver.py`

| Observação | Decisão |
|---|---|
| `VendorID`, `PULocationID`, `DOLocationID` mudam de `int64` para `int32` a partir de fevereiro | Padronizar para `int32` em todos os meses no Silver |
| `airport_fee` (janeiro) renomeada para `Airport_fee` (fev-mai) no Yellow | Padronizar para `airport_fee` minúsculo — não é coluna obrigatória, pode ser ignorada |
| `ehail_fee` muda de `object` para `float64` a partir de fevereiro no Green | Padronizar para `float64` — não é coluna obrigatória, pode ser ignorada |
| `passenger_count` é `float64` na fonte | Converter para `int32` no Silver — contagem de passageiros não pode ser decimal |
| Yellow usa `tpep_pickup_datetime` e `tpep_dropoff_datetime` | Renomear para `pickup_datetime` e `dropoff_datetime` no Silver |
| Green usa `lpep_pickup_datetime` e `lpep_dropoff_datetime` | Renomear para `pickup_datetime` e `dropoff_datetime` no Silver |
| 16 colunas em comum entre Yellow e Green | UNION ALL viável no Gold após padronização no Silver |
| Yellow tem 3 colunas exclusivas (`airport_fee`, `tpep_*`) | Ignorar no Silver — não são obrigatórias |
| Green tem 4 colunas exclusivas (`ehail_fee`, `lpep_*`, `trip_type`) | Ignorar no Silver — não são obrigatórias |
| Colunas obrigatórias do case presentes em ambas as tabelas | Todas as 5 colunas confirmadas na fonte |